<a href="https://colab.research.google.com/github/madhavcodes-07/Applied-machine-learning/blob/main/movie_recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install scikit-surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 17.0 MB/s eta 0:00:00


In [2]:
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip ml-latest-small.zip

--2026-07-13 05:31:54--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K  2.66MB/s    in 0.4s    

2026-07-13 05:31:54 (2.66 MB/s) - ‘ml-latest-small.zip’ saved [978202/978202]

Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  


In [3]:
import pandas as pd

ratings = pd.read_csv('ml-latest-small/ratings.csv')
movies = pd.read_csv('ml-latest-small/movies.csv')

print("Ratings shape:", ratings.shape)
print("Movies shape:", movies.shape)
ratings.head()

Ratings shape: (100836, 4)
Movies shape: (9742, 3)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [4]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

# Surprise library ke format mein data convert karo
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

# 80% training, 20% testing
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# SVD model banao aur train karo
model = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02)
model.fit(trainset)

print("Model trained successfully!")

Model trained successfully!


In [5]:
predictions = model.test(testset)

rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

RMSE: 0.8812
MAE:  0.6772


In [6]:
def recommend_movies(user_id, n=10):
    # Saari movies ki list
    all_movie_ids = movies['movieId'].unique()

    # Jo movies user ne already rate ki hain, unko chhodo
    rated_movies = ratings[ratings['userId'] == user_id]['movieId'].tolist()
    unrated = [m for m in all_movie_ids if m not in rated_movies]

    # Har unrated movie ke liye predicted rating nikalo
    predictions = [(m, model.predict(user_id, m).est) for m in unrated]

    # Highest predicted rating wali movies upar rakho
    predictions.sort(key=lambda x: x[1], reverse=True)
    top_n = predictions[:n]

    # Movie titles ke saath dikhao
    result = movies[movies['movieId'].isin([x[0] for x in top_n])][['movieId', 'title', 'genres']]
    return result

# Test karo - user ID 1 ke liye 10 recommendations
recommend_movies(user_id=1, n=10)

,movieId,title,genres
28,29,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi
277,318,"Shawshank Redemption, The (1994)",Crime|Drama
602,750,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War
659,858,"Godfather, The (1972)",Crime|Drama
680,898,"Philadelphia Story, The (1940)",Comedy|Drama|Romance
681,899,Singin' in the Rain (1952),Comedy|Musical|Romance
686,904,Rear Window (1954),Mystery|Thriller
692,910,Some Like It Hot (1959),Comedy|Crime
903,1201,"Good, the Bad and the Ugly, The (Buono, il bru...",Action|Adventure|Western
906,1204,Lawrence of Arabia (1962),Adventure|Drama|War


In [7]:
def get_popular_movies(n=10, min_ratings=50):
    # Har movie ke liye average rating aur total ratings count nikalo
    movie_stats = ratings.groupby('movieId').agg(
        avg_rating=('rating', 'mean'),
        rating_count=('rating', 'count')
    ).reset_index()

    # Sirf wahi movies lo jinko kaafi log rate kar chuke hain (warna 1 rating wali movie 5.0 dikha degi)
    popular = movie_stats[movie_stats['rating_count'] >= min_ratings]
    popular = popular.sort_values('avg_rating', ascending=False).head(n)

    result = popular.merge(movies, on='movieId')[['title', 'avg_rating', 'rating_count']]
    return result

# Test
print(get_popular_movies(n=10))

                                               title  avg_rating  rating_count
0                   Shawshank Redemption, The (1994)    4.429022           317
1                              Godfather, The (1972)    4.289062           192
2                                  Fight Club (1999)    4.272936           218
3                              Cool Hand Luke (1967)    4.271930            57
4  Dr. Strangelove or: How I Learned to Stop Worr...    4.268041            97
5                                 Rear Window (1954)    4.261905            84
6                     Godfather: Part II, The (1974)    4.259690           129
7                               Departed, The (2006)    4.252336           107
8                                  Goodfellas (1990)    4.250000           126
9                                  Casablanca (1942)    4.240000           100


In [8]:
def smart_recommend(user_id, n=10):
    if user_id in ratings['userId'].unique():
        print(f"Existing user {user_id} — personalized recommendations:")
        return recommend_movies(user_id, n)
    else:
        print(f"New user {user_id} — showing popular movies:")
        return get_popular_movies(n)

# Test dono cases
print(smart_recommend(user_id=1, n=5))      # existing user
print(smart_recommend(user_id=99999, n=5))  # naya user

Existing user 1 — personalized recommendations:
     movieId                                              title  \
28        29  City of Lost Children, The (Cité des enfants p...   
277      318                   Shawshank Redemption, The (1994)   
602      750  Dr. Strangelove or: How I Learned to Stop Worr...   
659      858                              Godfather, The (1972)   
680      898                     Philadelphia Story, The (1940)   

                                     genres  
28   Adventure|Drama|Fantasy|Mystery|Sci-Fi  
277                             Crime|Drama  
602                              Comedy|War  
659                             Crime|Drama  
680                    Comedy|Drama|Romance  
New user 99999 — showing popular movies:
                                               title  avg_rating  rating_count
0                   Shawshank Redemption, The (1994)    4.429022           317
1                              Godfather, The (1972)    4.289062          

In [9]:
from collections import defaultdict

def precision_recall_at_k(predictions, k=10, threshold=3.5):
    user_est_true = defaultdict(list)
    for pred in predictions:
        user_est_true[pred.uid].append((pred.est, pred.r_ui))

    precisions = {}
    recalls = {}

    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])
        n_rel_and_rec_k = sum(
            ((true_r >= threshold) and (est >= threshold))
            for (est, true_r) in user_ratings[:k]
        )

        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

    avg_precision = sum(prec for prec in precisions.values()) / len(precisions)
    avg_recall = sum(rec for rec in recalls.values()) / len(recalls)

    return avg_precision, avg_recall

precision, recall = precision_recall_at_k(predictions, k=10, threshold=3.5)
print(f"Precision@10: {precision:.3f}")
print(f"Recall@10: {recall:.3f}")

Precision@10: 0.744
Recall@10: 0.516


In [10]:
!pip install fastapi uvicorn pyngrok nest-asyncio


In [11]:
from fastapi import FastAPI
import nest_asyncio
from pyngrok import ngrok
import uvicorn

app = FastAPI()

@app.get("/")
def home():
    return {"message": "Movie Recommender API is running!"}

@app.get("/recommend/{user_id}")
def get_recommendations(user_id: int, n: int = 10):
    result = smart_recommend(user_id, n)
    return result.to_dict(orient="records")

In [17]:
from pyngrok import ngrok


ngrok.set_auth_token("3GR60JaYwnqn923QlTA8NYnN4IY_2FSVWFKf6eM7shFBNvo5n")

In [ ]:
import nest_asyncio
nest_asyncio.apply()
from pyngrok import ngrok

public_url = ngrok.connect(8000)
print("Your API is live at:", public_url)

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

Your API is live at: NgrokTunnel: "https://unturned-worsening-pointing.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [961]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2409:40d2:3004:b552:7c24:fc9:eb02:1f83:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d2:3004:b552:7c24:fc9:eb02:1f83:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     2409:40d2:3004:b552:7c24:fc9:eb02:1f83:0 - "GET / HTTP/1.1" 200 OK
Existing user 1 — personalized recommendations:


/usr/local/lib/python3.12/dist-packages/surprise/prediction_algorithms/algo_base.py:118: RuntimeWarning: coroutine 'Server.serve' was never awaited
  est = min(higher_bound, est)
ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-2' coro=<LifespanOn.main() running at /usr/local/lib/python3.12/dist-packages/uvicorn/lifespan/on.py:86> wait_for=<Future pending cb=[Task.__wakeup()]>>
ERROR:    Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/starlette/routing.py", line 645, in lifespan
    await receive()
GeneratorExit



INFO:     2409:40d2:3004:b552:7c24:fc9:eb02:1f83:0 - "GET /recommend/1 HTTP/1.1" 200 OK
